# Prep: picking diverse training samples with TF-IDF + UMAP + cosine similarity

Builds TF-IDF + lexicon features on a 10% sample, reduces them with UMAP and keeps the 50 least similar articles (the idea later reused for non-attack undersampling).

In [ ]:
DATA_DIR = "../data"

In [ ]:
!pip install nlp-id --q
!pip install nltk --q
!pip install umap-learn[plot] --q
!pip install catboost --q
!pip install sentence_transformers --q
!pip install transformers --q
!pip install evaluate --q
!pip install accelerate --q

In [ ]:
import nltk
import warnings

warnings.filterwarnings('ignore')
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import time
import umap
import umap.plot
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
import keras.backend as K

from transformers import BertTokenizer, TFBertModel

from catboost import CatBoostClassifier

from nlp_id.lemmatizer import Lemmatizer
from nlp_id.tokenizer import Tokenizer, PhraseTokenizer
from nlp_id.postag import PosTag
from nlp_id.stopword import StopWord

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold
from sklearn import feature_selection as fs
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn import preprocessing
from sklearn.metrics.pairwise import cosine_similarity

from scipy import sparse

from sentence_transformers import SentenceTransformer

from IPython.display import IFrame
from IPython.core.display import display, HTML

In [ ]:
# !wget -c https://lumbung.cs.ui.ac.id/f/0a606ecd51d84a94b224/?dl=1 -O gemastik_22.csv
!wget https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv
!wget https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv

--2023-09-10 14:20:37--  https://lumbung.cs.ui.ac.id/f/0a606ecd51d84a94b224/?dl=1
Resolving lumbung.cs.ui.ac.id (lumbung.cs.ui.ac.id)... 152.118.148.86
Connecting to lumbung.cs.ui.ac.id (lumbung.cs.ui.ac.id)|152.118.148.86|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://lumbung.cs.ui.ac.id/seafhttp/files/a0ec6030-cc7e-48d2-8def-161c463bcd7f/subset.csv [following]
--2023-09-10 14:20:39--  https://lumbung.cs.ui.ac.id/seafhttp/files/a0ec6030-cc7e-48d2-8def-161c463bcd7f/subset.csv
Reusing existing connection to lumbung.cs.ui.ac.id:443.
HTTP request sent, awaiting response... 200 OK
Length: 22944526 (22M) [application/octet-stream]
Saving to: ‘gemastik_22.csv’

gemastik_22.csv     100%[===================>]  21.88M  3.18MB/s    in 17s     

2023-09-10 14:20:56 (1.28 MB/s) - ‘gemastik_22.csv’ saved [22944526/22944526]

--2023-09-10 14:20:56--  https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv
Resolving raw.githubusercontent.com (raw.gi

In [ ]:
data_main = pd.read_csv(f"{DATA_DIR}/gemastik_22.csv")

# sample saja
data = data_main.sample(frac = 0.10, replace = False, random_state = 1)
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 529 entries, 4532 to 123
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                529 non-null    object
 1   sumber            529 non-null    object
 2   kodekat           529 non-null    object
 3   kodesubkat        529 non-null    object
 4   kategori          529 non-null    object
 5   subkategori       529 non-null    object
 6   katakunci         529 non-null    object
 7   tanggal           529 non-null    object
 8   judul_berita      529 non-null    object
 9   konten_berita     529 non-null    object
 10  nama_tokoh        529 non-null    object
 11  jabatan           529 non-null    object
 12  organisasi        529 non-null    object
 13  lokasi            529 non-null    object
 14  alias             529 non-null    object
 15  kutipan           529 non-null    object
 16  sentimen_kutipan  529 non-null    object
 17  sentimen_beri

In [ ]:
stopwords = StopWord().get_stopword()
lemmatizer = Lemmatizer()
tokenizer = Tokenizer()
postagger = PosTag()
ph_tokenizer = PhraseTokenizer()

def tokenize_lemmatize(text):
  lemmatized_doc = lemmatizer.lemmatize(text)
  return [word for word in tokenizer.tokenize(lemmatized_doc) if word not in stopwords]

In [ ]:
def load_sentword(filename):
  file = open(filename, "r")
  file.readline() # skip header
  sentword_score = {}
  for line in file:
    word, score = line.strip().split("\t")
    score = int(score)
    sentword_score[word] = score
  file.close()
  return sentword_score

sentword_score_neg = load_sentword("negative.tsv")
sentword_score_pos = load_sentword("positive.tsv")

# num positives or negatives
def num(text, sentword_score):
  count = 0
  for word in text.split():
    if word in sentword_score:
      count += 1
  return count

# num JJ positives or negatives
def num_jj(text, sentword_score):
  count = 0
  for word, pos in postagger.get_pos_tag(text):
    if pos == "JJ":
      if word in sentword_score:
        count += 1
  return count

In [ ]:
# Ekstraksi Fitur

data["lemmatized_konten"] = data.konten_berita.\
                               map(lambda x: ' '.join(tokenize_lemmatize(x)))

data["lemmatized_judul"] = data.judul_berita.\
                               map(lambda x: ' '.join(tokenize_lemmatize(x)))

# fitur AUX 1
data["num_pos_konten"] = data.lemmatized_konten.\
                               map(lambda x: num(x, sentword_score_pos))

# fitur AUX 2
data["num_neg_konten"] = data.lemmatized_konten.\
                               map(lambda x: num(x, sentword_score_neg))

# fitur AUX 3
data["num_jj_pos_konten"] = data.lemmatized_konten.\
                               map(lambda x: num_jj(x, sentword_score_pos))

# fitur AUX 4
data["num_jj_neg_konten"] = data.lemmatized_konten.\
                               map(lambda x: num_jj(x, sentword_score_neg))

In [ ]:
#TF-IDF judul dan konten
data["judul_konten"] = data.apply(lambda x: x['lemmatized_judul'] + " " + x['lemmatized_konten'], axis = 1)

vectorizer = TfidfVectorizer(min_df = 3, max_df = .6, ngram_range = (1,2)) #make Tfidf Vectorizer
vectorizer.fit(data["judul_konten"])

tfidf_vecs = []
for tfidf_vec in vectorizer.transform(data["judul_konten"]).toarray():
  tfidf_vecs.append(tfidf_vec)

data['tfidf_judul_konten'] = tfidf_vecs

In [ ]:
len(list(data["tfidf_judul_konten"])[0])

7895

In [ ]:
# sampai sini, kita gabung 4 fitur aux + 7895 fitur TFIDF
# total ada 7899 feature, yang akan kita reduksi dimensinya dengan UMAP

all_features = []
for index, row in data.iterrows():
  feature = []
  feature += list(row["tfidf_judul_konten"])
  feature.append(row["num_pos_konten"])
  feature.append(row["num_neg_konten"])
  feature.append(row["num_jj_pos_konten"])
  feature.append(row["num_jj_neg_konten"])
  all_features.append(feature)

all_features = np.array(all_features)
print(all_features.shape)

reduced_features = umap.UMAP(n_components = 64, metric = 'hellinger').fit_transform(all_features)

(529, 7899)


In [ ]:
# pairwise cosine similarity

#reduced_features_sparse = sparse.csr_matrix(all_features)
reduced_features_sparse = sparse.csr_matrix(reduced_features)
similarities = cosine_similarity(reduced_features_sparse)

#print(similarities.shape)
pairwise = np.mean(similarities, axis = 1)

# urutan instances dari yang memiliki mean cosine paling rendah hingga paling besar
sorted_index = np.argsort(pairwise)
print(sorted_index)

[ 74 273 450 518  41 102 340   1  56 302 291 388 332  33 494 223 460 486
 272 224 100  13  98 262 294 342 493  54 152 155 290 501 269 140 429 257
 316  80 396 168 212 458 141 339 323 434  10 183 159 165   0 221 337 242
  15 500 177 237 348 521 480 478  19 275 127  97 104 347 525 354  94 514
 411 191  49  14 181 184 112  51  35 198 452 115  39 477 444  23 219 265
  17 190 160 194   9 358 263  86  88  53 362  71 412 313  21  36 210 403
 455 387  40 503  75 369 402 491  55 284 410 169 331 246 164 317   6  12
  72  44 173 205 413  76 426 225 409 270 338  93 254 267 473 449 176 139
 243 226 289 167 507 158 522 130 326 485 330 128 451 407 415 122 443  85
 114 131 353 400 107 285 180 319 328  42   3 453 432 116 361 110 497 133
 504 527 151 367 105 488  95 195 373 109 281 471 142 153 238 282  65 108
 371 427   4 459 385 343 192 356 251 232 466 288 227 123 187 216 197 274
 124   5 327 322 276 457 271 352  89 389  37 437 247 303  90  64 416 508
 222  70 256 315 495 359 430 435 366 509  59 189  1

In [ ]:
sorted(sorted_index[:50])

[1,
 10,
 13,
 33,
 41,
 54,
 56,
 74,
 80,
 98,
 100,
 102,
 140,
 141,
 152,
 155,
 159,
 165,
 168,
 183,
 212,
 223,
 224,
 257,
 262,
 269,
 272,
 273,
 290,
 291,
 294,
 302,
 316,
 323,
 332,
 339,
 340,
 342,
 388,
 396,
 429,
 434,
 450,
 458,
 460,
 486,
 493,
 494,
 501,
 518]

In [ ]:
# pilih top 50 instances dengan rata-rata cosine paling rendah (paling tidak mirip dengan yang lainnya)
filtered_data = data.iloc[sorted_index[:50], :]

filtered_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 50 entries, 5107 to 4109
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  50 non-null     object
 1   sumber              50 non-null     object
 2   kodekat             50 non-null     object
 3   kodesubkat          50 non-null     object
 4   kategori            50 non-null     object
 5   subkategori         50 non-null     object
 6   katakunci           50 non-null     object
 7   tanggal             50 non-null     object
 8   judul_berita        50 non-null     object
 9   konten_berita       50 non-null     object
 10  nama_tokoh          50 non-null     object
 11  jabatan             50 non-null     object
 12  organisasi          50 non-null     object
 13  lokasi              50 non-null     object
 14  alias               50 non-null     object
 15  kutipan             50 non-null     object
 16  sentimen_kutipan    50 